# Project 03 (final): explaining and predicting demand

**Scenario (continuing from module 02):** the management of Capital Bikeshare was
delighted with your EDA. Now they want more:

1. **Explain:** how strongly does daily demand depend on weather and calendar —
   in numbers? (→ regression)
2. **Predict:** how many bikes will be needed tomorrow? (→ time series forecast)

**Data: real** — the same bike sharing data (UCI) as in module 02, this time at the
**daily level** (`day.csv`, 731 days 2011-2012). Preparation (once):

```
python datasets/download_data.py
```

**Reference to the script:** sections 1.1-1.2 (regression), 2.1 (time series),
2.2 (feature engineering, leakage!).

## 1. Loading the data

De-normalisation as before (`temp*41` = degrees C etc. — reading the documentation pays off).

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 2. Simple regression: demand ~ temperature

**Tasks:**
1. Fit `cnt ~ temp_c` with `LinearRegression` (careful: sklearn expects X as 2D —
   `df[["temp_c"]]`, not `df["temp_c"]`).
2. Print the slope, intercept and $R^2$ and formulate the slope as a sentence
   ("per degree more ...").
3. Draw a scatter plot plus the regression line.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 3. Residual analysis — keeping an eye on the model

**Task:** plot the residuals ($y - \hat{y}$) against the temperature.
Do you see a curvature? What does it mean substantively? (Script 1.1: residual plot!)

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Finding:** an inverted U shape — at mild temperatures the line underestimates,
on very hot days it overestimates. Substantively clear: **above about 30 degrees, cycling
stops being fun.** A straight line cannot represent "first up, then down".

**Task:** add `temp_c2 = temp_c**2` as a feature and fit `cnt ~ temp_c + temp_c2`.
Compare $R^2$ and compute the **vertex** $-\beta_1 / (2\beta_2)$ — the computed
comfort temperature.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 4. Multiple regression: the explanatory model

**Task:** fit `cnt ~ temp_c + temp_c2 + hum_pct + wind_kmh + workingday + yr`
and interpret EVERY coefficient in one sentence — **ceteris paribus** (script 1.2!).
`yr` is particularly interesting (0 = 2011, 1 = 2012): what does it measure?

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Reference interpretation (compare with yours):**

- `hum_pct` about −32: per percentage point of humidity, about 32 fewer rentals (other conditions equal)
- `wind_kmh` about −77: wind is a clear deterrent
- `workingday` about +98: working days bring somewhat more volume at the daily level (the commuters!)
- `yr` about +1893: **in 2012 the days were on average about 1900 rentals above 2011** — this
  is the growth trend as a number. Without `yr` this trend would "seep" into the other
  coefficients (omitted variable bias!)
- Interpreting `temp_c`/`temp_c2` individually is pointless (the multicollinearity between
  $x$ and $x^2$ is intended here) — they act only *together*, as a parabola.

**A warning against causal talk:** "+98 on working days" is a *description* of the data,
not an experiment. For causal statements we lack confounder control and randomisation.

## 5. Forecasting: knowing in the morning what will be missing tonight

Now the switch from **explaining** to **predicting** — with the rules from script 2.1/2.2:

- **Lag features:** `lag1` (yesterday), `lag7` (a week ago), `roll7`
  (mean of the last 7 days, shifted by one day — why that needs the `.shift(1)` is the
  leakage question below!)
- **Temporal split:** training up to 30 September 2012, test from 1 October 2012 (92 days)
- **Baselines first:** naive (= yesterday) and seasonal naive (= 7 days ago)
- **Metric:** MAE (mean absolute error — directly interpretable in "bikes")

**Task:** build the three lag features, split temporally, compute the MAE of the two
baselines and then train a `LinearRegression` on weather + calendar + lags. Does it beat
the baselines?

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Look at the worst days:** at the end of October 2012 demand collapses to almost
zero — that is **Hurricane Sandy** (29/30 October 2012, Washington D.C. came to a
standstill). No model in the world predicts such a thing from lag features. Real forecasts
therefore need external information (weather warnings, events) — and honest error figures
that include such days.

## 6. The leakage demo: why the temporal split is not negotiable

**Task:** train the same model with a **random** 75/25 split (`train_test_split`,
`random_state=0`) and compare the test MAE with your temporal split. Explain the
difference. (Script 2.2: leakage!)

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Explanation:** with the random split, training days lie temporally BETWEEN the
test days — for a test day the model often knows the previous and the following day from
training, and the lag features carry their information straight in. The roughly 625 are
not forecasting performance but a measurement artefact. The honest value is the temporal
split (about 860): **that is how good we really would have been had we forecast every day
from October 2012 onwards.**

## 7. Conclusion for management

**Task:** again 4-5 bullet points in everyday language: what drives demand (with numbers!),
how well can we predict tomorrow, where are the limits?

<details><summary>Reference conclusion</summary>

- Temperature is the strongest lever: demand rises up to about 28 degrees and falls above that — hot days are NOT peak days.
- Wind and high humidity noticeably depress demand (about 77 and about 32 rentals per unit respectively).
- The business grew in 2012 by about 1,900 rentals per day on average compared with 2011 — capacity planning has to extrapolate this trend.
- Our daily forecast is on average about 860 bikes off (with 2,000-8,000 rentals per day) and beats simple rules of thumb ("like yesterday": about 950).
- Limits: extreme events (Hurricane Sandy) cannot be predicted from historical patterns — such days need weather warnings in the process.
</details>

## Done — what you can do now

- regression as an explanatory tool: read coefficients ceteris paribus, place $R^2$ in
  context, check residuals, capture non-linearity with a quadratic term
- regression as a forecasting tool: lag features, temporal split, baselines, MAE
- not just name leakage but **demonstrate it in numbers**
- communicate the limits of a model honestly (Sandy!)

**Bonus tasks:**
1. Sin/cos encoding of the month (script 2.2) instead of `yr` + season dummies — does it help?
2. Compute the MAE separately for October/November/December — does the forecast get worse towards the end of the year? Why might that be?
3. Standardise the features and compare the coefficients — which variable now has the largest magnitude, and why is that the fairer basis for comparison?

---

**Module 03 is thereby complete.** Next comes Machine Learning 1 — where "fitting a
regression" becomes a systematic toolbox with validation, regularisation and many more
model families.